#Import Library


In [41]:
import pandas as pd
import numpy as np

import random
import hashlib


#Load Dataset EPO from GitHub

##Load Dataset/Label

In [42]:
anger_url = 'https://raw.githubusercontent.com/Ricco48/Emotion-Dataset-from-Indonesian-Public-Opinion/refs/heads/main/Emotion%20Dataset%20from%20Indonesian%20Public%20Opinion/AngerData.csv'
fear_url = 'https://raw.githubusercontent.com/Ricco48/Emotion-Dataset-from-Indonesian-Public-Opinion/refs/heads/main/Emotion%20Dataset%20from%20Indonesian%20Public%20Opinion/FearData.csv'
joy_url = 'https://raw.githubusercontent.com/Ricco48/Emotion-Dataset-from-Indonesian-Public-Opinion/refs/heads/main/Emotion%20Dataset%20from%20Indonesian%20Public%20Opinion/JoyData.csv'
sad_url = 'https://raw.githubusercontent.com/Ricco48/Emotion-Dataset-from-Indonesian-Public-Opinion/refs/heads/main/Emotion%20Dataset%20from%20Indonesian%20Public%20Opinion/SadData.csv'
# love_url = 'https://raw.githubusercontent.com/Ricco48/Emotion-Dataset-from-Indonesian-Public-Opinion/refs/heads/main/Emotion%20Dataset%20from%20Indonesian%20Public%20Opinion/LoveData.csv'
neutral_url = 'https://raw.githubusercontent.com/Ricco48/Emotion-Dataset-from-Indonesian-Public-Opinion/refs/heads/main/Emotion%20Dataset%20from%20Indonesian%20Public%20Opinion/NeutralData.csv'

df_anger = pd.read_csv(anger_url, sep='\t')
df_fear = pd.read_csv(fear_url, sep='\t')
df_joy = pd.read_csv(joy_url, sep='\t')
df_sad = pd.read_csv(sad_url, sep='\t')
# df_love = pd.read_csv(love_url, sep='\t')
df_neutral = pd.read_csv(neutral_url, sep='\t')

##Merge Dataset EPO

In [43]:
df_epo = pd.concat([df_anger, df_fear, df_joy, df_sad, df_neutral], ignore_index=True)
df_epo['Label'] = df_epo['Label'].str.lower()

##Rename Label Dataset

In [44]:
df_epo = df_epo.rename(columns={'Tweet': 'text', 'Label': 'label'})
df_epo.head()

,text,label
0,pagi2 udah di buat emosi :),anger
1,"kok stabilitas negara, memange 10 thn negara t...",anger
2,dah lah emosi mulu liat emyu,anger
3,"aib? bodoh benar! sebelum kata aib itu muncul,...",anger
4,dih lu yg nyebelin bego,anger


##Mapping Label

In [45]:
epo_map = {
    'joy': 'happy',
    # 'love' : 'happy',
    'fear' : 'anxious',
    'neutral' : 'neutral',
    'sad' : 'sad',
    'anger' : 'angry'
}

df_epo['emotion_label'] = df_epo['label'].map(epo_map)

##Distribusi Label

In [46]:
df_epo['emotion_label'].value_counts()

,count
emotion_label,
neutral,2001
happy,1275
angry,1130
sad,1003
anxious,911


##Mapping Stress Score Berdasarkan Emosi

In [47]:
stress_range = {
    'happy': (0.0, 0.2),
    'neutral': (0.2, 0.4),
    'sad': (0.5, 0.8),
    'anxious': (0.7, 0.9),
    'angry': (0.85, 1.0)
}

In [48]:
def generate_stress_score(text, label):
    if label not in stress_range:
        return None

    low, high = stress_range[label]

    seed = int(hashlib.md5(text.encode()).hexdigest(), 16) % (10**8)
    random.seed(seed)

    score = random.uniform(low, high)
    return round(score, 2)

In [49]:
df_epo['stress_label'] = df_epo.apply(lambda x: generate_stress_score(x['text'], x['emotion_label']), axis=1)

In [50]:
df_epo = df_epo[['text', 'emotion_label', 'stress_label']]
df_epo.head()

,text,emotion_label,stress_label
0,pagi2 udah di buat emosi :),angry,0.90
1,"kok stabilitas negara, memange 10 thn negara t...",angry,0.93
2,dah lah emosi mulu liat emyu,angry,0.95
3,"aib? bodoh benar! sebelum kata aib itu muncul,...",angry,0.88
4,dih lu yg nyebelin bego,angry,1.00


#Load Sythentic Dataset


In [51]:
df_llm = pd.read_csv('https://raw.githubusercontent.com/raihanmeintaro/Dataset/refs/heads/main/NLP_Dataset.csv')
df_llm.head()

,text,emotion_label,stress_label
0,anxious banget sebelum meeting tadi sih di kan...,anxious,0.80
1,kehilangan semangat buat ngapa ngapain tapi ya...,sad,0.64
2,takut gagal di presentasi besok tapi yaudahlah...,anxious,0.81
3,akhir minggu ini seru banget serius tadi pagi ...,happy,0.16
4,makan favorit hari ini bikin bahagia wkwk sama...,happy,0.11


##Rename Label Dataset

In [52]:
df_llm['emotion_label'] = df_llm['emotion_label'].replace({'fear': 'anxious'})
df_llm['emotion_label'].value_counts()

,count
emotion_label,
anxious,5000
sad,5000
happy,5000
angry,5000
neutral,5000


##Distribusi Label

In [53]:
df_llm['emotion_label'].value_counts()

,count
emotion_label,
anxious,5000
sad,5000
happy,5000
angry,5000
neutral,5000


#Merge All Dataset

In [54]:
df_combined = pd.concat([df_epo, df_llm], ignore_index=True)
df_combined.shape

(31320, 3)

In [55]:
df_combined['emotion_label'].value_counts()

,count
emotion_label,
neutral,7001
happy,6275
angry,6130
sad,6003
anxious,5911


##Convert Dataset

In [56]:
df_combined.to_csv('df_combined.csv')